Styling: seaborn's "ticks" theme (clean, minimal, journal-friendly) is used for
every bar/line/heatmap chart. Pies and stacked bars use matplotlib directly
(seaborn has no native chart types for these) but inherit the same seaborn
theme, fonts and color palette so the whole figure set looks consistent.
A colour-blind-safe qualitative palette (Okabe-Ito) is used throughout, with a
separate distinguishable grey ramp reserved for "no info" style categories
(Not specified / No / Unclear / No comparison made / Not reported) so those
never collapse into a single indistinguishable grey.

In [4]:
!pip install seaborn

In [5]:


import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------------
# 0. Setup & global theme
# ------------------------------------------------------------------
DATA_PATH = "QML_benchmarking_cleaned.xlsx"
OUT_DIR = "figures_sb"
os.makedirs(OUT_DIR, exist_ok=True)

sns.set_theme(
    style="ticks",
    context="paper",
    font_scale=1.2,
    rc={
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",
        "axes.edgecolor": "#4D4D4D",
        "axes.labelcolor": "#2B2B2B",
        "text.color": "#2B2B2B",
        "xtick.color": "#2B2B2B",
        "ytick.color": "#2B2B2B",
        "axes.grid": True,
        "grid.color": "#E6E6E6",
        "grid.linewidth": 0.6,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.titlepad": 12,
    },
)

PALETTE = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7",
           "#56B4E9", "#F0E442", "#7F3C8D"]

# Distinct grey shades reserved for "no information" style categories, so grey
# still reads as "no info" family at a glance, but each shade of "no" stays
# visually distinguishable in legends / stacked bars (darkest = most negative).
NEUTRAL_PALETTE = ["#BFBFBF", "#8C8C8C", "#595959", "#333333"]
NEUTRAL_LABELS = {"not specified", "not reported", "no", "unclear", "other",
                   "no comparison made", "no / unclear"}


def ordered_colors(labels):
    """Assign PALETTE colors to `labels` in order, but map any label that
    matches NEUTRAL_LABELS (case-insensitive) to a distinct grey shade from
    NEUTRAL_PALETTE instead of a normal palette color, so 'no info' categories
    stay visually related (all greys) while still being distinguishable from
    each other. Cycles through each palette if there are more labels than
    available colors."""
    colors = []
    palette_cycle = itertools.cycle(PALETTE)
    neutral_cycle = itertools.cycle(NEUTRAL_PALETTE)
    for lab in labels:
        if str(lab).strip().lower() in NEUTRAL_LABELS:
            colors.append(next(neutral_cycle))
        else:
            colors.append(next(palette_cycle))
    return colors


df = pd.read_excel(DATA_PATH, sheet_name="Papers")


def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("saved", path)


# ------------------------------------------------------------------
# Reusable chart helpers
# ------------------------------------------------------------------
def pie(series, title, filename, min_pct_for_label=3, colors=None):
    """Donut chart (matplotlib — seaborn has no pie chart type), styled to
    match the seaborn theme used everywhere else."""
    counts = series.value_counts()
    colors = colors or ordered_colors(counts.index)
    fig, ax = plt.subplots(figsize=(7, 6))
    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=None,
        autopct=lambda p: f"{p:.0f}%" if p >= min_pct_for_label else "",
        startangle=90,
        colors=colors[: len(counts)],
        pctdistance=0.75,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=1.5),
        textprops=dict(color="white", fontweight="bold", fontsize=10),
    )
    ax.legend(
        wedges,
        [f"{idx} (n={v})" for idx, v in counts.items()],
        title=title,
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
    )
    ax.set_title(title, fontweight="bold")
    save(fig, filename)


def hbar(counts, xlabel, title, filename, colors=None, figsize=None, sort=True):
    """Horizontal bar chart via sns.barplot, ordered largest-to-smallest."""
    if sort:
        counts = counts.sort_values(ascending=False)
    colors = colors or ordered_colors(counts.index)
    palette = dict(zip(counts.index, colors))
    figsize = figsize or (9, max(2.5, 0.5 * len(counts) + 1))
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        x=counts.values, y=counts.index, hue=counts.index,
        order=counts.index, hue_order=counts.index,
        palette=palette, legend=False, ax=ax, edgecolor="white", linewidth=0.6,
    )
    for i, v in enumerate(counts.values):
        ax.text(v + max(counts.values) * 0.012, i, int(v), va="center", fontsize=9.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.set_title(title)
    sns.despine(ax=ax, left=True)
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save(fig, filename)


def stacked_bar(ct_pct, ylabel, title, filename, rotate=15, figsize=(9, 5.5)):
    """100%-stacked bar chart (matplotlib — seaborn has no native stacked-bar
    support), styled with the same theme/palette/greys as everything else."""
    colors = ordered_colors(ct_pct.columns)
    fig, ax = plt.subplots(figsize=figsize)
    bottom = np.zeros(len(ct_pct))
    for i, col in enumerate(ct_pct.columns):
        ax.bar(ct_pct.index.astype(str), ct_pct[col].values, bottom=bottom,
               label=col, color=colors[i], edgecolor="white", linewidth=0.7, width=0.65)
        bottom += ct_pct[col].values
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9.5)
    plt.setp(ax.get_xticklabels(), rotation=rotate, ha="right" if rotate else "center")
    sns.despine(ax=ax)
    fig.tight_layout()
    save(fig, filename)


# ------------------------------------------------------------------
# 1. Pie chart — Data nature (Data_Type_Category)
# ------------------------------------------------------------------
pie(df["Data_Type_Category"], "Distribution of Data Modalities Used", "01_data_type_pie.png")

# ------------------------------------------------------------------
# 2. Papers per year — bar + cumulative line (classic "growth" chart)
# ------------------------------------------------------------------
year_counts = df["Year"].value_counts().sort_index()
years_str = year_counts.index.astype(str)
fig, ax1 = plt.subplots(figsize=(8, 5))
sns.barplot(x=years_str, y=year_counts.values, color=PALETTE[0], ax=ax1, edgecolor="white")
ax1.set_ylabel("Number of papers", color=PALETTE[0])
ax1.set_xlabel("Year")
ax1.set_title("Number of QML Benchmarking Papers per Year")
for i, v in enumerate(year_counts.values):
    ax1.text(i, v + 0.2, str(v), ha="center", fontsize=9.5)

ax2 = ax1.twinx()
cum = year_counts.cumsum()
sns.lineplot(x=range(len(years_str)), y=cum.values, color=PALETTE[1], marker="o",
             linewidth=2.2, markersize=7, ax=ax2)
ax2.set_ylabel("Cumulative number of papers", color=PALETTE[1])
ax2.grid(False)
sns.despine(ax=ax1)
sns.despine(ax=ax2, right=False)
fig.tight_layout()
save(fig, "02_papers_per_year.png")

# ------------------------------------------------------------------
# 3. Contribution type — new method proposal vs benchmark-only vs both
# ------------------------------------------------------------------
hbar(df["Contribution_Type"].value_counts(), "Number of papers",
     "Type of Contribution: Proposed Methods vs. Benchmark Studies", "03_contribution_type.png")

# ------------------------------------------------------------------
# 4. Real vs synthetic datasets
# ------------------------------------------------------------------
pie(df["Dataset_Nature"], "Dataset Nature: Real vs. Synthetic", "04_dataset_nature_pie.png")

# ------------------------------------------------------------------
# 5. Real-world application or not
# ------------------------------------------------------------------
pie(df["Real_World_Application"], "Targets a Real-World Application?", "05_real_world_application_pie.png")

# ------------------------------------------------------------------
# 6. Library used (primary library)
# ------------------------------------------------------------------
pie(df["Primary_Library"], "Primary Software Library / Framework Used", "06_primary_library_pie.png")

# 6b. Bonus: since many papers use several libraries, also show co-occurrence
lib_cols = [c for c in df.columns if c.startswith("Uses_") and c not in ("Uses_Multiple_Libraries",)]
lib_usage = (df[lib_cols] == "Yes").sum().sort_values(ascending=False)
lib_usage.index = [c.replace("Uses_", "") for c in lib_usage.index]
fig, ax = plt.subplots(figsize=(8.5, 5))
sns.barplot(x=lib_usage.index, y=lib_usage.values, color=PALETTE[1], ax=ax, edgecolor="white")
ax.set_ylabel("Number of papers mentioning the library")
ax.set_xlabel("")
ax.set_title("Software Library Usage (papers can use more than one)")
plt.setp(ax.get_xticklabels(), rotation=40, ha="right")
for i, v in enumerate(lib_usage.values):
    ax.text(i, v + 0.2, int(v), ha="center", fontsize=9.5)
sns.despine(ax=ax)
fig.tight_layout()
save(fig, "06b_library_usage_bar.png")

# ------------------------------------------------------------------
# 7. Simulator vs real quantum computer
# ------------------------------------------------------------------
pie(df["Hardware_Type"], "Simulator vs. Real Quantum Hardware", "07_hardware_type_pie.png")

# ------------------------------------------------------------------
# 8.  Quantum advantage reported (overall)
# ------------------------------------------------------------------
pie(df["Quantum_Advantage_Reported"], "Reported Quantum Advantage over Classical Baselines",
    "08_quantum_advantage_pie.png")

# ------------------------------------------------------------------
# 9.  Task category distribution
# ------------------------------------------------------------------
hbar(df["Task_Category"].value_counts(), "Number of papers",
     "Machine Learning Task Addressed", "09_task_category_bar.png",
     colors=[PALETTE[2]] * df["Task_Category"].nunique())

# ------------------------------------------------------------------
# 10.  Data type x Task heatmap (combine two columns)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Data_Type_Category"], df["Task_Category"])
task_order = ct.sum(axis=0).sort_values(ascending=False).index
ct = ct[task_order]
annot = ct.astype(str)
annot[ct == 0] = ""
fig, ax = plt.subplots(figsize=(10, 5.5))
sns.heatmap(
    ct, annot=annot, fmt="", cmap=sns.light_palette(PALETTE[0], as_cmap=True),
    linewidths=1, linecolor="white", cbar_kws={"label": "Number of papers", "shrink": 0.8},
    annot_kws={"fontsize": 9.5}, ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)
ax.set_title("Data Modality vs. ML Task (paper counts)")
fig.tight_layout()
save(fig, "10_datatype_vs_task_heatmap.png")

# ------------------------------------------------------------------
# 11.  Quantum advantage vs Hardware type (stacked bar, %)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Hardware_Type"], df["Quantum_Advantage_Reported"], normalize="index") * 100
stacked_bar(ct, "% of papers", "Reported Quantum Advantage by Hardware Type",
            "11_advantage_by_hardware_stacked.png")

# ------------------------------------------------------------------
# 12.  Papers per year, stacked by data modality (trend over time)
# ------------------------------------------------------------------
ct = pd.crosstab(df["Year"], df["Data_Type_Category"])
ct = ct.reindex(sorted(ct.index))
stacked_bar(ct, "Number of papers", "Publication Trend by Data Modality",
            "12_trend_by_modality_stacked.png", rotate=0)

# ------------------------------------------------------------------
# 13.  Has public code available (open-science indicator)
# ------------------------------------------------------------------
pie(df["Has_Public_Code"], "Public Code Repository Available", "13_has_code_pie.png")

# ------------------------------------------------------------------
# 14.  Top journals / venues publishing this literature
# ------------------------------------------------------------------
journal_counts = df["Journal"].value_counts().head(12)
hbar(journal_counts, "Number of papers", "Top Venues Publishing QML Benchmarking Papers",
     "14_top_journals_bar.png", colors=[PALETTE[0]] * len(journal_counts), sort=False)

# ------------------------------------------------------------------
# 15.  Common critique / limitation themes (keyword-mined from "Critic")
# ------------------------------------------------------------------
CRITIQUE_THEMES = {
    "Noise / decoherence": r"noise|decoheren|error rate",
    "Small dataset / subset": r"small dataset|subset|only \d|limited data|130 sentences|525 points",
    "Scalability limits": r"scalab|qubit limit|qubit count|high-dimensional|cannot handle",
    "No / weak classical baseline": r"no compar|not compared|no classical|without classical",
    "Fixed / simple circuits": r"fixed circuit|circuits are fixed|generic approach|simple",
    "Limited hardware access": r"real hardware|physical hardware|nisq|hardware noise",
    "Reporting issues (train vs test, etc.)": r"training accuracy|not test accuracy|architetcure is not shown|not shown",
    "Future work / early-stage": r"future work|early exploratory|promising but limited",
}

critic_text = df["Critic"].dropna().str.lower()
theme_counts = {
    theme: critic_text.str.contains(pattern, regex=True, na=False).sum()
    for theme, pattern in CRITIQUE_THEMES.items()
}
theme_counts = pd.Series(theme_counts)
theme_counts = theme_counts[theme_counts > 0]
hbar(theme_counts, f"Number of papers (out of {critic_text.shape[0]} with a recorded critique)",
     "Most Common Reported Limitations / Critiques", "15_critique_themes_bar.png",
     colors=[PALETTE[4]] * len(theme_counts))

# ------------------------------------------------------------------
# 16.  Evaluation metrics used (keyword-mined from "metrics")
# ------------------------------------------------------------------
METRIC_FAMILIES = {
    "Accuracy": r"accuracy",
    "Precision / Recall": r"precision|recall",
    "F1-score": r"f1",
    "AUC / ROC": r"auc|roc",
    "Loss": r"\bloss\b",
    "MSE / RMSE": r"\bmse\b|\brmse\b",
    "MAE / MAPE": r"\bmae\b|\bmape\b",
    "R\u00b2": r"r2|r\u00b2",
    "Execution / training time": r"time|runtime",
}
metrics_text = df["metrics"].dropna().str.lower()
metric_counts = {
    fam: metrics_text.str.contains(pat, regex=True, na=False).sum()
    for fam, pat in METRIC_FAMILIES.items()
}
metric_counts = pd.Series(metric_counts)
hbar(metric_counts, f"Number of papers (out of {metrics_text.shape[0]} with recorded metrics)",
     "Evaluation Metrics Used Across Studies", "16_metrics_used_bar.png",
     colors=[PALETTE[2]] * len(metric_counts))

# ------------------------------------------------------------------
# 17.  Contribution type vs. quantum advantage (publication-bias check)
#     Do "new method proposal" papers report advantage more often than
#     pure benchmark studies that have no stake in a specific new model?
# ------------------------------------------------------------------
ct = pd.crosstab(df["Contribution_Type"], df["Quantum_Advantage_Reported"], normalize="index") * 100
ct = ct.reindex(ct.sum(axis=1).sort_values(ascending=False).index)
stacked_bar(ct, "% of papers",
            "Reported Quantum Advantage by Type of Contribution\n(possible publication-framing bias check)",
            "17_advantage_by_contribution_type_stacked.png", rotate=20, figsize=(9.5, 5.5))

# ------------------------------------------------------------------
# 18.  Dataset nature (real vs synthetic) vs. quantum advantage
# ------------------------------------------------------------------
ct = pd.crosstab(df["Dataset_Nature"], df["Quantum_Advantage_Reported"], normalize="index") * 100
stacked_bar(ct, "% of papers", "Reported Quantum Advantage by Dataset Nature",
            "18_advantage_by_dataset_nature_stacked.png")

# ------------------------------------------------------------------
# 19.  Benchmarking rigor over time (avg. # methods benchmarked / year)
# ------------------------------------------------------------------
rigor = df.groupby("Year")["Num_Methods_Benchmarked"].mean()
rigor = rigor.reindex(sorted(rigor.index))
years_str = rigor.index.astype(str)
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(x=range(len(years_str)), y=rigor.values, marker="o", linewidth=2.2,
             markersize=7, color=PALETTE[6], ax=ax)
ax.fill_between(range(len(rigor)), rigor.values, alpha=0.15, color=PALETTE[6])
ax.set_xticks(range(len(years_str)))
ax.set_xticklabels(years_str)
ax.set_ylabel("Avg. number of methods benchmarked")
ax.set_xlabel("Year")
ax.set_title("Benchmarking Rigor Over Time")
for i, v in enumerate(rigor.values):
    ax.text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=9.5)
sns.despine(ax=ax)
fig.tight_layout()
save(fig, "19_rigor_over_time_line.png")

print("\nAll figures written to:", os.path.abspath(OUT_DIR))

saved figures_sb\01_data_type_pie.png
saved figures_sb\02_papers_per_year.png
saved figures_sb\03_contribution_type.png
saved figures_sb\04_dataset_nature_pie.png
saved figures_sb\05_real_world_application_pie.png
saved figures_sb\06_primary_library_pie.png
saved figures_sb\06b_library_usage_bar.png
saved figures_sb\07_hardware_type_pie.png
saved figures_sb\08_quantum_advantage_pie.png
saved figures_sb\09_task_category_bar.png
saved figures_sb\10_datatype_vs_task_heatmap.png
saved figures_sb\11_advantage_by_hardware_stacked.png
saved figures_sb\12_trend_by_modality_stacked.png
saved figures_sb\13_has_code_pie.png
saved figures_sb\14_top_journals_bar.png
saved figures_sb\15_critique_themes_bar.png
saved figures_sb\16_metrics_used_bar.png
saved figures_sb\17_advantage_by_contribution_type_stacked.png
saved figures_sb\18_advantage_by_dataset_nature_stacked.png
saved figures_sb\19_rigor_over_time_line.png

All figures written to: d:\QML benchmarking\Visualizations_QML_survey\figures_sb
